# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print a summary overview
meta = dataset.metadata
meta_json = meta.to_json()  # for easier inspection
print(f"{meta_json['name']}: {meta_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate the record sets and show the available fields and their `@id`s.

In [ ]:
# Get all record sets in the dataset and their fields
record_sets = dataset.record_sets
print("Available Record Sets and Fields:")
record_set_ids = []

for rs in record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', rs_id)
    print(f"\nRecord Set @id: {rs_id}\n  Name: {rs_name}")
    record_set_ids.append(rs_id)
    # List all fields in this record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - Field @id: {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

for rs_id in record_set_ids:
    records_gen = dataset.records(record_set=rs_id)
    records = list(records_gen)
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set: {rs_id} with shape: {df.shape}")
    else:
        print(f"Record set '{rs_id}' has no record data or is not tabular.")

# Pick the first actual tabular record set for inspection (change if needed)
selected_rs = None
for rs_id in dataframes:
    selected_rs = rs_id
    break

if selected_rs is not None:
    print(f'Columns in record set {selected_rs}:')
    print(list(dataframes[selected_rs].columns))
    display(dataframes[selected_rs].head())
else:
    print("No tabular record sets were found to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You should choose a numeric field and a field for grouping from the output of above, using their `@id` values.

In [ ]:
# Choose record_set, numeric field, and group field by their @id (replace as appropriate)
record_set_id = selected_rs  # Example record set id (should be chosen from overview above)
df = dataframes[record_set_id]

# Try to find a numeric field, like 'Coverage_percent' or 'MW_kDa' by common names or pick by @id
import numpy as np

# For demonstration, try to detect the first numeric column (change as appropriate)
numeric_field = None
for col in df.columns:
    # Heuristic: treat as numeric if most values are numbers
    vals = pd.to_numeric(df[col], errors='coerce')
    if np.isfinite(vals).sum() > 0.5 * len(df):
        numeric_field = col
        break
if numeric_field is None:
    print("No numeric fields detected for EDA.")
else:
    print(f"Selected numeric field: {numeric_field}")

    # Filter records for values > threshold (e.g. 10)
    threshold = 10
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    vals = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    filtered_df[f"{numeric_field}_normalized"] = (vals - vals.mean()) / vals.std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a candidate group field (first non-numeric str column)
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            # Only select if few unique values (e.g. like, protein function/class)
            if df[col].nunique() < 0.5 * len(df):
                break
    if group_field:
        print(f"Grouping by {group_field} (first listed string field)")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, visualize mean by group
    if group_field:
        plt.figure(figsize=(10, 5))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(20)
        sns.barplot(y=group_means.index, x=group_means.values, orient='h')
        plt.title(f"Top 20 mean {numeric_field} by {group_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we successfully loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the Croissant schema and `mlcroissant`. We identified the available record sets and fields, loaded tabular data, performed basic cleaning and normalization, explored numeric field distributions, and visualized feature statistics. This workflow can be extended for deeper domain analysis or integration into larger FAIR/ML pipelines. To perform targeted biology/biochemistry analysis, continue by refining field selection and specifying thresholds/aggregations aligned with experimental attributes.